# imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

import os
import json

# data set-up

In [ ]:
tstate = 0
np.random.seed(tstate)

# (˶˃ᆺ˂˶) !!!!!!!!!! dataset = r"dataset file path"
# (˶˃ᆺ˂˶) !!!!!!!!!! outputdir = r"output directory file path"
os.makedirs(outputdir,exist_ok=True)

#projection metric + PCA component no.
outcome = "updrs_3"
principeptiden = 10

# data clean-up

In [ ]:
#retrieving dataset
pddata = pd.read_csv(dataset)
pddata = pddata.copy()

print(f"total visits: {pddata.shape[0]}")
print(f"total patients: {pddata["patient_id"].nunique()}")

#create visit year column
pddata["visit_year"] = pddata["visit_month"]/12. 

#separate clinical data from proteomic data
clinicald = ["visit_id", "patient_id", "visit_month", "visit_year", "updrs_1", "updrs_2", "updrs_3", "updrs_4", "updrs_total", "upd23b_clinical_state_on_medication"]
peptided = [p for p in pddata.columns if p not in clinicald]

print(f"\nclinical data: {len(clinicald)} columns")
print(f"peptide data: {len(peptided)} columns")

#create new df cutting visits with no recorded UPDRS-3 score
fulld = len(pddata)
pddata = pddata.dropna(subset=[outcome]).copy()

print(f"\ncut {fulld-len(pddata)} visits with missing {outcome}") 
print(f"left with {len(pddata)} visits, {pddata["patient_id"].nunique()} patients")

# patients grouped by visit medication state

In [ ]:
pmedstate = pddata.groupby("patient_id")["upd23b_clinical_state_on_medication"].apply(lambda μ: set(μ.dropna()))
pmedstate

#sort into classes
pclass = {}
for pid, medstate in pmedstate.items():
    if len(medstate) == 0:
        pclass[pid] = "unrecorded"
    elif medstate == {"Off"}:
        pclass[pid] = "control"
    elif medstate == {"On"}:
        pclass[pid] = "treated"
    else:
        pclass[pid] = "joint"

pddata["treatment_class"] = pddata["patient_id"].map(pclass)

classcount = pddata.groupby("treatment_class")["patient_id"].nunique()
print(classcount)
print(f"\ncontrol = off medication, \njoint = on and off medication recorded, \ntreated = always on medication, \nunrecorded = no medication data") 

# training control grp = off-meds + no med data

In [ ]:
#control + treated patient grps
controlps = set(pid for pid, ε in pclass.items() if ε in ("control", "unrecorded"))
treatedps = set(pid for pid, φ in pclass.items() if φ in ("treated", "joint"))

print(f"\ncontrol (for training): {len(controlps)} patients, \ntreated (ever \"On\" meds): {len(treatedps)} patients")

#yes/no for training grp in dataset
pddata["trainingps"] = pddata["patient_id"].apply(lambda ω: "yes" if ω in controlps else "no")
pddata

# extracting patient baseline (patient earliest visit)

In [ ]:
#patient first visit distribution
baseline = pddata.sort_values("visit_month").groupby("patient_id").first().reset_index()
print(f"baseline month dist: {baseline["visit_month"].value_counts().sort_index().to_dict()}")

# random split of control grp into train + test grps

In [ ]:
#control grp baseline df + split 80/20 into train/test grps
ctrlbaseline = baseline[baseline["patient_id"].isin(controlps)]
trainps, testps = train_test_split(list(controlps),test_size=0.2,random_state=tstate)

trainps = set(trainps)
testps = set(testps)

print(f"\n{len(trainps)} train patients, \n{len(testps)} test patients, \n{len(treatedps)} not included (treated) patients")

# baseline peptide imputing, scaling, + PCA

## evaluating missingness before imputing + splitting patient baseline peptide profiles into train/test/treated groups

In [ ]:
#baseline full peptide profile + UPDRS-3 scores
baselinepeps = baseline.set_index("patient_id")[peptided]
baselineoutcome = baseline.set_index("patient_id")[outcome]

print(f"baseline peptide profile dataframe (patients, peptides): {baselinepeps.shape}")
print(f"missing values per peptide (fraction): mean={baselinepeps.isnull().mean().mean():.3f}, max={baselinepeps.isnull().mean().max():.3f}")

#assess missingness
missingrate = baselinepeps.isnull().mean()
ζ = 0.3
pepoverζ = (missingrate > ζ).sum()
print(f"\n{pepoverζ} peptide(s) with over 30% missingness")

#remove peptide(s) with >30% missingness
keep = missingrate[missingrate <= ζ].index
baselinepeps = baselinepeps[keep]
print(f"removed {pepoverζ} peptide(s) from patient peptide profiles")

#split baseline into train/test/treated grps
trainbs = baselinepeps.loc[baselinepeps.index.isin(trainps)]
testbs = baselinepeps.loc[baselinepeps.index.isin(testps)]
treatedbs = baselinepeps.loc[baselinepeps.index.isin(treatedps)]

print(f"\nbaselines (patients, peptides) ; \ntrain: {trainbs.shape}, test: {testbs.shape}, treated: {treatedbs.shape}") 

## imputing

In [9]:
imp = SimpleImputer(strategy="median")

trainimp = pd.DataFrame(imp.fit_transform(trainbs),index=trainbs.index,columns=baselinepeps.columns)
testimp = pd.DataFrame(imp.transform(testbs),index=testbs.index,columns=baselinepeps.columns)
treatedimp = pd.DataFrame(imp.transform(treatedbs),index=treatedbs.index,columns=baselinepeps.columns)
allimp = pd.DataFrame(imp.transform(baselinepeps),index=baselinepeps.index,columns=baselinepeps.columns)

## scaling

In [10]:
scaler = StandardScaler()

scaledtrain = pd.DataFrame(scaler.fit_transform(trainimp),index=trainimp.index,columns=baselinepeps.columns)
scaledtest = pd.DataFrame(scaler.transform(testimp),index=testimp.index,columns=baselinepeps.columns)
scaledtreated = pd.DataFrame(scaler.transform(treatedimp),index=treatedimp.index,columns=baselinepeps.columns)
scaledall = pd.DataFrame(scaler.transform(allimp),index=allimp.index,columns=baselinepeps.columns)

## PCA

In [ ]:
#principal component analysis
pca = PCA(n_components=principeptiden,random_state=tstate)

trainpca = pca.fit_transform(scaledtrain)
testpca = pca.transform(scaledtest)
treatedpca = pca.transform(scaledtreated)
allpca = pca.transform(scaledall)

#get cumulative, total + per-component variance explained ratio
print(f"PCA variance explained ratio ({principeptiden} components)")
for η, ver in enumerate(pca.explained_variance_ratio_):
    print(f"PC{η+1} {ver:.3f} (cum: {pca.explained_variance_ratio_[:η+1].sum():.3f})")

print(f"\ntotal: {pca.explained_variance_ratio_.sum():.3f}")

#baseline PC dfs
pcd = [f"PC{η+1}" for η in range(principeptiden)]
pctrain = pd.DataFrame(trainpca,index=trainbs.index,columns=pcd)
pctest = pd.DataFrame(testpca,index=testbs.index,columns=pcd)
pctreated = pd.DataFrame(treatedpca,index=treatedbs.index,columns=pcd)
pcall = pd.DataFrame(allpca,index=baselinepeps.index,columns=pcd)

# digi-twin model longitudinal modelling table

In [ ]:
#attach patient baseline PCs + UPDRS-3 scores to dataset
modelframe = []
for _, row in pddata.iterrows():
    pid = row["patient_id"]
    if pid not in pcall.index:
        continue

    visitrecord = {
        "patient_id": pid,
        "visit_month": row["visit_month"],
        "visit_year": row["visit_year"],
        "updrs_3": row["updrs_3"],
        "baseline_updrs_3": baselineoutcome.loc[pid],
        "training_patient": row["trainingps"],
        "split": "train" if pid in trainps else ("test" if pid in testps else "treated"),
    }

    for pc in pcd:
        visitrecord[f"baseline_{pc}"] = pcall.loc[pid,pc]

    modelframe.append(visitrecord)

#final digi-twin modelling data
dtpmodeld = pd.DataFrame(modelframe)

print(f"digi-twin prototype longitudinal modelling table (visits, columns): {dtpmodeld.shape}")
print(f"\npatients: {dtpmodeld["patient_id"].nunique()}")
print(f"split counts (patients)")
print(dtpmodeld.groupby("split")["patient_id"].nunique())
print(f"\nsplit counts (visits)")
print(dtpmodeld["split"].value_counts())

dtpmodeld.to_csv(os.path.join(outputdir,"digi-twin_prototype_LMT.csv"),index=False)
print(f"\nsaved {outputdir}\\digi-twin_prototype_LMT.csv")

# UPDRS-3 score trajectory spaghetti graphs (control + treated)

In [ ]:
plt.rcParams["font.family"]="Times New Roman"
plt.rcParams["font.size"]=10.5

fig, spag = plt.subplots(1, 2, figsize=(7,2.5))

#control
ctrld = dtpmodeld[dtpmodeld["split"].isin(["train", "test"])]
for pid in ctrld["patient_id"].unique():
    ptimeline = ctrld[ctrld["patient_id"] == pid].sort_values("visit_year")
    spag[0].plot(ptimeline["visit_year"],ptimeline["updrs_3"],alpha=0.3,color="#743089",linewidth=0.8,label="control" if pid == ctrld["patient_id"].unique() [0] else "")

spag[0].set_xlabel("time from baseline (years)")
spag[0].set_ylabel("UPDRS-III score")
spag[0].set_xlim(0,10)
spag[0].set_ylim(0,80)

spag[0].spines["top"].set_visible(False)
spag[0].spines["right"].set_visible(False)

spag[0].legend(loc="upper right", frameon=False)

#treated
trd = dtpmodeld[dtpmodeld["split"] == "treated"]
for pid in trd["patient_id"].unique():
    ptimeline = trd[trd["patient_id"] == pid].sort_values("visit_year")
    spag[1].plot(ptimeline["visit_year"],ptimeline["updrs_3"],alpha=0.3,color="#d0557a",linewidth=0.8,label="treated" if pid == trd["patient_id"].unique() [0] else "")

spag[1].set_xlabel("time from baseline (years)")
spag[1].set_ylabel("UPDRS-III score")
spag[1].set_xlim(0,8)
spag[1].set_ylim(0,80)

spag[1].spines["top"].set_visible(False)
spag[1].spines["right"].set_visible(False)

spag[1].legend(loc="upper right", frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(outputdir,"updrs3_trajectories.png"),dpi=150)
plt.close()
print(f"\nsaved {outputdir}\\updrs3_trajectories.png")

# baseline vs. follow-up UPDRS-3 score scatter plot

## is baseline UPDRS-3 score a reliable predictor of follow-up UPDRS-3 score?

In [ ]:
plt.rcParams["font.family"]="Times New Roman"
plt.rcParams["font.size"]=11

fig, uc = plt.subplots(figsize=(6,5))
followup = dtpmodeld[dtpmodeld["visit_year"]>0]
uc.scatter(followup["baseline_updrs_3"],followup["updrs_3"],alpha=0.3,s=20,color="#743089")

uc.set_xlabel("baseline UPDRS-III score")
uc.set_ylabel("follow-up UPDRS-III score")
uc.set_xlim(0,80)
uc.set_ylim(0,90)

r = np.corrcoef(followup["baseline_updrs_3"],followup["updrs_3"])[0,1]

uc.text(0.95,0.95,f"r={r:.2f}",transform=uc.transAxes,ha="right",va="bottom",fontsize=10)

uc.spines["top"].set_visible(False)
uc.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(outputdir,"updrs3_baseline_vs_followup.png"),dpi=150)
plt.close()
print(f"saved {outputdir}\\updrs3_baseline_vs_followup.png")

# baseline peptide PCA by grp scatter plot

In [ ]:
plt.rcParams["font.family"]="Times New Roman"
plt.rcParams["font.size"]=11

fig, psp = plt.subplots(figsize=(7,5))
for split, color, label in [("train", "#00d1bf", "control (train)"),("test", "#0068d1", "control (test)"),("treated", "#d10092", "treated")]:
    jatp = dtpmodeld[dtpmodeld["split"] == split].drop_duplicates("patient_id")
    psp.scatter(jatp["baseline_PC1"],jatp["baseline_PC2"],alpha=0.6,s=30,c=color,label=label)

psp.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
psp.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
psp.set_xlim(None,60)
psp.set_ylim(None,30)

psp.legend(loc="upper right")

psp.spines["top"].set_visible(False)
psp.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(outputdir,"baseline_PCA_scatter.png"),dpi=150)
plt.close()
print(f"saved {outputdir}\\baseline_PCA_scatter.png")

# baseline UPDRS-3 score per grp summary

In [ ]:
#mean +- SD of baseline UPDRS-3 scores for train/test/treated grps 
for split in ["train","test","treated"]:
    jatp = dtpmodeld[dtpmodeld["split"] == split]
    print(f"{split}: ({len(jatp)} visits, {jatp["patient_id"].nunique()} patients) baseline UPDRS-3 score: mean={jatp["updrs_3"].mean():.2f}, sd={jatp["updrs_3"].std():.2f}")

# longitudinal mixed-effects model

## training no-treatment forecaster + applying to all patients + evaluating model performance

In [ ]:
#longitudinal mixed-effects model dfs
ftrain = dtpmodeld[dtpmodeld["split"] == "train"].copy()
ftest = dtpmodeld[dtpmodeld["split"] == "test"].copy()
ftreated = dtpmodeld[dtpmodeld["split"] == "treated"].copy()

#regression formula
mmpc = " + ".join([f"baseline_PC{δ+1}" for δ in range(5)])
rf = f"updrs_3 ~ visit_year + baseline_updrs_3 + {mmpc} + visit_year:baseline_updrs_3"
print(f"regression formula: {rf} \n(outcome ~ time + baseline + baseline*time + covariates(peptides/interaction so effect of time is based on baseline UPDRS-3))")
print(f"\ntraining data: {len(ftrain)} visits, {ftrain["patient_id"].nunique()} patients")

#fitting mixed-effects model
#try fit model with random intercept + slope, else fit with just random intercept
try:
    mmd = smf.mixedlm(rf,data=ftrain,groups=ftrain["patient_id"],re_formula="~visit_year")
    mfit = mmd.fit(method="lbfgs",maxiter=2000)
    print(f"\nlongitudinal mixed-effects model converged: {mfit.converged}")
    print(mfit.summary().tables[0])
    print(mfit.summary().tables[1])
    mm = mfit
    convergence = True
except Exception as β:
    print(f"random slope failed ({β}), fitting with random intercept only")
    mmd = smf.mixedlm(rf,data=ftrain,groups=ftrain["patient_id"],re_formula="1")
    mfit = mmd.fit(method="lbfgs",maxiter=2000)
    print(f"\nlongitudinal mixed-effects model (random intercept only) converged: {mfit.converged}")
    print(mfit.summary().tables[0])
    print(mfit.summary().tables[1])
    mm = mfit
    convergence = False

#computing UPDRS-3 forecast values
def forecast(model,data):
    fes = model.fe_params
    fullrf = model.model.formula
    rhs = fullrf.split("~",1)[1].strip()
    X = patsy.dmatrix(rhs,data,return_type="dataframe")
    return X @ fes

ftest["forecast"] = forecast(mfit,ftest)
ftreated["forecast"] = forecast(mfit,ftreated)
ftrain["forecast"] = forecast(mfit,ftrain)

#model performance
def perfcheck(recorded,predicted,group):
    rmse = np.sqrt(mean_squared_error(recorded,predicted))
    mae = mean_absolute_error(recorded,predicted)
    r2 = r2_score(recorded,predicted)
    print(f"{group}: RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")
    return{"rmse":rmse,"mae":mae,"r2":r2}

print(f"\nlongitudinal mixed-effects model performance check")
perftrain = perfcheck(ftrain["updrs_3"],ftrain["forecast"],"control (train)")
perftest = perfcheck(ftest["updrs_3"],ftest["forecast"],"control (test)")
perftreated = perfcheck(ftreated["updrs_3"],ftreated["forecast"],"treated")

# feedforward neural network

## training no-treatment forecaster + applying to all patients + evaluating model performance

In [ ]:
#inputs
nin = ["baseline_updrs_3", "visit_year"] + [f"baseline_PC{α+1}" for α in range(principeptiden)]

#input/output matrices
trainin = ftrain[nin].values
trainpred = ftrain["updrs_3"].values

testin = ftest[nin].values
testpred = ftest["updrs_3"].values

treatedin = ftreated[nin].values
treatedpred = ftreated["updrs_3"].values

#scaling
nscaler = StandardScaler()
strainin = nscaler.fit_transform(trainin)
stestin = nscaler.transform(testin)
streatedin = nscaler.transform(treatedin)

print(f"feedforward neural network input features: {len(nin)}")
print(f"features: {nin}")
print(f"\n(visits, features) ; train: {strainin.shape}, test: {stestin.shape}, treated: {streatedin.shape}")

#training
mlp = MLPRegressor(hidden_layer_sizes=(64,32),activation="relu",solver="adam",alpha=0.01,max_iter=2000,early_stopping=True,validation_fraction=0.15,n_iter_no_change=30,random_state=tstate,verbose=False)
mlp.fit(strainin,trainpred)
print(f"\nMLP trained ({mlp.n_iter_} iterations), final loss={mlp.loss_:.4f}")

#computing UPDRS-3 forecast values
ftrain["nforecast"] = mlp.predict(strainin)
ftest["nforecast"] = mlp.predict(stestin)
ftreated["nforecast"] = mlp.predict(streatedin)

#model performance
print(f"\nfeedforward neural network performance check")
nperftrain = perfcheck(ftrain["updrs_3"],ftrain["nforecast"],"control (train)")
nperftest = perfcheck(ftest["updrs_3"],ftest["nforecast"],"control (test)")
nperftreated = perfcheck(ftreated["updrs_3"],ftreated["nforecast"],"treated")

# mixed-effects model vs. feedforward neural network

## which forecaster is better?

In [ ]:
#df
mvsf = pd.DataFrame({"mixed-effects": [perftrain,perftest,perftreated], "feedforward neural network": [nperftrain,nperftest,nperftreated]},index=["control (train)","control (test)","treated"])

#RMSE + MAE comparison
print(f"\nRMSE comparison:")
for split in mvsf.index:
    print(f"{split}: mixed-effects={mvsf.loc[split,"mixed-effects"]["rmse"]:.3f}, feedforward neural network={mvsf.loc[split,"feedforward neural network"]["rmse"]:.3f}")

print(f"\nMAE comparison:")
for split in mvsf.index:
    print(f"{split}: mixed-effects={mvsf.loc[split,"mixed-effects"]["mae"]:.3f}, feedforward neural network={mvsf.loc[split,"feedforward neural network"]["mae"]:.3f}")

## predicted vs. recorded UPDRS-3 score scatter plots for both models

In [ ]:
plt.rcParams["font.family"]="Times New Roman"
plt.rcParams["font.size"]=11

fig, pvsr = plt.subplots(3,2,figsize=(12,15))

for c, (model,pred) in enumerate([("mixed-effects","forecast"), ("feedforward neural network","nforecast")]):
    for r, (group,df) in enumerate([("control (train)",ftrain), ("control (test)",ftest), ("treated",ftreated)]):
        pvr = pvsr[r,c]
        pvr.scatter(df[pred],df["updrs_3"],alpha=0.4,s=20,color="#743089")
        lims = [0,max(df["updrs_3"].max(),df[pred].max())+5]
        
        pvr.plot(lims,lims,"r--",alpha=0.5)
        pvr.set_xlabel("predicted UPDRS-III")
        pvr.set_ylabel("recorded UPDRS-III")
        pvr.set_xlim(0,None)
        pvr.set_ylim(0,None)

        rmse = np.sqrt(mean_squared_error(df["updrs_3"],df[pred]))

        pvr.set_title(f"{group} ({model})\nRMSE={rmse:.1f}")

        pvr.spines["top"].set_visible(False)
        pvr.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(outputdir,"pvr_mvsf.png"),dpi=150,bbox_inches="tight")
plt.close()
print(f"\nsaved {outputdir}\\pvr_mvsf.png")

# model efficacy visualisation (on random sample of test + treated patients)

In [ ]:
plt.rcParams["font.family"]="Times New Roman"
plt.rcParams["font.size"]=11

fig, sample = plt.subplots(2,3,figsize=(15,10))
sampp = list(ftest["patient_id"].unique())[:3] + list(ftreated["patient_id"].unique())[:3]

for κ,pid in enumerate(sampp):
    r,c = κ//3, κ%3
    ex = sample[r,c]

    #retrieve all visits for each patient
    pdata = dtpmodeld[dtpmodeld["patient_id"] == pid].sort_values("visit_year")
    split = pdata["split"].iloc[0]

    ex.plot(pdata["visit_year"],pdata["updrs_3"],"ko-",markersize=5,label="recorded")

    #sort predictions by group
    if pid in ftrain["patient_id"].values:
        sampd = ftrain[ftrain["patient_id"] == pid].sort_values("visit_year")
    elif pid in ftest["patient_id"].values:
        sampd = ftest[ftest["patient_id"] == pid].sort_values("visit_year")
    else:
        sampd = ftreated[ftreated["patient_id"] == pid].sort_values("visit_year")

    ex.plot(sampd["visit_year"],sampd["forecast"],color="#743089",markersize=4,label="mixed-effects model",alpha=0.8)
    ex.plot(sampd["visit_year"],sampd["nforecast"],color="#0068d1",markersize=4,label="feedforward neural network",alpha=0.8)

    ex.set_xlabel("time from baseline (years)")
    ex.set_ylabel("UPDRS-III score")
    ex.legend(fontsize=8)
    ex.set_xlim(0,None)
    ex.set_ylim(0,80)

    ex.spines["top"].set_visible(False)
    ex.spines["right"].set_visible(False)

    fig.text(-0.02,0.77,"control (test)",va="center",ha="left",rotation=90,fontsize=12,fontweight="bold")
    fig.text(-0.02,0.28,"treated",va="center",ha="left",rotation=90,fontsize=12,fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(outputdir,"sample_trajectories.png"),dpi=150,bbox_inches="tight")
plt.close()
print(f"saved {outputdir}\\sample_trajectories.png")

# 5-fold patient-grouped cross-validation

## evaluating how well either model predicts UPDRS-3 trajectories on "new" control patients

In [ ]:
#fold n
gkf = GroupKFold(n_splits=5)

cv = {"mrmse":[], "mmae":[], "nrmse":[], "nmae":[]}

#creating patient groups
ctrldf = dtpmodeld[dtpmodeld["split"].isin(["train","test"])].copy()

for fold, (traini,vali) in enumerate(gkf.split(ctrldf,groups=ctrldf["patient_id"])):
    trainfo = ctrldf.iloc[traini]
    valfo = ctrldf.iloc[vali]

    #mxied-effects model
    #try fit model with just random intercept (simplified for small folds), else fit with fixed-effects (OLS)
    try:
        frf = f"updrs_3 ~ visit_year + baseline_updrs_3 + {mmpc}"
        fmmd = smf.mixedlm(frf,data=trainfo,groups=trainfo["patient_id"],re_formula="1")
        fmmdfit = fmmd.fit(method="lbfgs",maxiter=1000)
        
        frhs = frf.split("~",1)[1].strip()
        fXv = patsy.dmatrix(frhs,valfo,return_type="dataframe")
        valforecast = (fXv @ fmmdfit.fe_params).values

        cv["mrmse"].append(np.sqrt(mean_squared_error(valfo["updrs_3"],valforecast)))
        cv["mmae"].append(mean_absolute_error(valfo["updrs_3"],valforecast))

    except Exception as ξ:
        print(f"fold {fold+1}: mixed-effects failed ({ξ}), fitting with OLS (fixed-effects only)")
        try:
            frfOLS = f"updrs_3 ~ visit_year + baseline_updrs_3 + {mmpc}"
            ftraininOLS = patsy.dmatrix(frfOLS.split("~",1)[1].strip(),trainfo,return_type="dataframe")
            fvalOLS = patsy.dmatrix(frfOLS.split("~",1)[1].strip(),valfo,return_type="dataframe")
            
            ols = sm.OLS(trainfo["updrs_3"].values,ftraininOLS).fit()
            valforecast = ols.predict(fvalOLS).values

            cv["mrmse"].append(np.sqrt(mean_squared_error(valfo["updrs_3"],valforecast)))
            cv["mmae"].append(mean_absolute_error(valfo["updrs_3"],valforecast))

        except Exception as ξ2:
            print(f"fold {fold + 1}: OLS failed ({ξ2})")
            cv["mrmse"].append(np.nan)
            cv["mmae"].append(np.nan)

    #feedforward neural network
    trainfops = set(trainfo["patient_id"].unique())
    fbaseline = baseline[baseline["patient_id"].isin(trainfops)]

    #re-do PCA (fit to train)
    fimp = SimpleImputer(strategy="median")
    fintrain = fbaseline.set_index("patient_id")[baselinepeps.columns]
    fintrimp = pd.DataFrame(fimp.fit_transform(fintrain),index=fintrain.index,columns=baselinepeps.columns)
    fmscaler = StandardScaler()
    sfintr = fmscaler.fit_transform(fintrimp)
    fpca = PCA(n_components=principeptiden,random_state=tstate)
    fpca.fit(sfintr)

    #impute and scale all ctrl patients
    allbasef = baseline[baseline["patient_id"].isin(controlps)]
    inallf = allbasef.set_index("patient_id")[baselinepeps.columns]
    inallfimp = pd.DataFrame(fimp.transform(inallf),index=inallf.index,columns=baselinepeps.columns)
    sinall = fmscaler.transform(inallfimp)
    fpcs = fpca.transform(sinall)
    fpcsdf = pd.DataFrame(fpcs,index=inallf.index,columns=[f"baseline_PC{ρ+1}" for ρ in range(principeptiden)])

    #building fold model data + feedforward neural network fit and forecasting
    fmodeld = ctrldf.copy()
    for fpc in [f"baseline_PC{ρ+1}" for ρ in range(principeptiden)]:
        fmodeld[fpc] = fmodeld["patient_id"].map(fpcsdf[fpc])

    intrf = fmodeld.iloc[traini][nin].values
    predtrf = fmodeld.iloc[traini]["updrs_3"].values
    invalf = fmodeld.iloc[vali][nin].values
    predvalf = fmodeld.iloc[vali]["updrs_3"].values

    fnscaler = StandardScaler()
    intrfsc = fnscaler.fit_transform(intrf)
    invalfsc = fnscaler.transform(invalf)

    fmlp = MLPRegressor(hidden_layer_sizes=(64,32),activation="relu",solver="adam",alpha=0.01,max_iter=2000,early_stopping=True,validation_fraction=0.15,n_iter_no_change=30,random_state=tstate)
    fmlp.fit(intrfsc,predtrf)
    fnpred = fmlp.predict(invalfsc)

    cv["nrmse"].append(np.sqrt(mean_squared_error(predvalf,fnpred)))
    cv["nmae"].append(mean_absolute_error(predvalf,fnpred))

    print(f"fold {fold+1}: mixed-effects model RMSE={cv["mrmse"][-1]:.2f}, feedforward neural network RMSE={cv["nrmse"][-1]:.2f}")

print(f"\ncross-val summary:")
print(f"mixed-effects model: RMSE={np.nanmean(cv["mrmse"]):.3f} ± {np.nanstd(cv["mrmse"]):.3f}, MAE={np.nanmean(cv["mmae"]):.3f} ± {np.nanstd(cv["mmae"]):.3f}")
print(f"feedforward neural network: RMSE={np.nanmean(cv["nrmse"]):.3f} ± {np.std(cv["nrmse"]):.3f}, MAE={np.mean(cv["mmae"]):.3f} ± {np.std(cv["mmae"]):.3f}")

## mean RMSE per fold bar chart for both models

In [ ]:
plt.rcParams["font.family"]="Times New Roman"
plt.rcParams["font.size"]=11

fig, cvp = plt.subplots(figsize=(8,5))
folds = range(1,6)
Ω = np.arange(len(folds))
width = 0.35
cvp.bar(Ω-width/2,cv["mrmse"],width,label="mixed-effects model",color="#743089",alpha=0.8)
cvp.bar(Ω+width/2,cv["nrmse"],width,label="feedforward neural network",color="#0068d1",alpha=0.8)
cvp.set_xlabel("cross-val fold")
cvp.set_ylabel("mean RMSE")
cvp.set_ylim(0,10)

cvp.set_xticks(Ω)
cvp.set_xticklabels(f"{n}" for n in folds)
cvp.legend()

cvp.spines["top"].set_visible(False)
cvp.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(outputdir,"cross-val-comp.png"),dpi=150)
plt.close()
print(f"saved {outputdir}\\cross_val_comp.png")

# output summary + clean dataset dls

In [ ]:
#output overview
output = {
    "modelframe": {
        "total_patients": int(pddata["patient_id"].nunique()),
        "total_visits": len(pddata),
        "all_peptide_seqs": len(peptided),
        "filtered_pep_seqs": len(baselinepeps.columns),
        "outcome": outcome,
        "control_patients": len(controlps),
        "treated_patients": len(treatedps),
        "train_patients": len(trainps),
        "test_patients": len(testps),
        "baseline_patients": len(baseline),
        "max_followup_month": int(pddata["visit_month"].max())
    },
    "pca": {
        "components": principeptiden,
        "tot_variance_explained_ratio": float(pca.explained_variance_ratio_.sum()),
        "per_c_VER": [float(Θ) for Θ in pca.explained_variance_ratio_],
    },
    "mixed_effects_model": {
        "regression_formula": rf,
        "converged": bool(mfit.converged),
        "random_slope": convergence,
        "train": perftrain,
        "test": perftest,
        "treated": perftreated,
        "cross_val_mean_rmse": float(np.nanmean(cv["mrmse"])),
        "cross_val_rmse_SD": float(np.nanstd(cv["mrmse"])),
    },
    "feedforward_neural_network": {
        "architecture": [64,32],
        "inputs": nin,
        "iterations": int(mlp.n_iter_),
        "final_loss": float(mlp.loss_),
        "train": nperftrain,
        "test": nperftest,
        "treated": nperftreated,
        "cross_val_mean_rmse": float(np.mean(cv["nrmse"])),
        "cross_val_rmse_SD": float(np.std(cv["nrmse"])),
    },
}

#create json
with open(os.path.join(outputdir,"dtp_model_setup.json"), "w") as Γ:
    json.dump(output,Γ,indent=2)
print(f"saved {outputdir}\\dtp_model_setup.json")

#cleaned data dl
dtpmodeld.to_csv(os.path.join(outputdir,"dtpmodel_data.csv"),index=False)
baseline.to_csv(os.path.join(outputdir,"baseline_data.csv"),index=False)
pcall.to_csv(os.path.join(outputdir,"baseline_pca.csv"),index=False)
print(f"saved dtpmodel_data.csv, baseline_data.csv, baseline_pca.csv to {outputdir}")

print(f"\nall outputs saved to {outputdir}")
print(f"files:")
for υ in sorted(os.listdir(outputdir)):
    print(f"{υ}")